In [11]:
import torch
import torch.nn.functional as F

import torch
import torch

def get_confusion_matrix(
    logits: torch.Tensor,   # (N, C, ...)
    target: torch.Tensor,   # (N, ...)
    use_logits: bool = True,
    ignore_index = None,
) -> torch.Tensor:
    C = int(logits.shape[1])
    preds = (torch.softmax(logits, dim=1).argmax(dim=1) if use_logits
             else logits.argmax(dim=1))
    y = target
    if ignore_index is not None:
        mask = (y != ignore_index)
        preds, y = preds[mask], y[mask]
    y = y.reshape(-1).to(torch.int64)
    preds = preds.reshape(-1).to(torch.int64)
    valid = (y >= 0) & (y < C) & (preds >= 0) & (preds < C)
    if valid.numel() == 0 or valid.sum() == 0:
        return torch.zeros((C, C), dtype=torch.long, device=logits.device)
    y, preds = y[valid], preds[valid]
    cm = torch.zeros((C, C), dtype=torch.long, device=logits.device)
    idx = y * C + preds
    counts = torch.bincount(idx, minlength=C*C)
    cm += counts.view(C, C)
    return cm


def confusion_matrix_update_corners(
    M: torch.Tensor,                  # (C, C)
    logits: torch.Tensor,             # (N, C, ...)
    target: torch.Tensor,             # (N, ...)
    alpha: float = 0.05,
    use_logits: bool = True,
    ignore_index = None,
    weights = None,                   # (C, C)
):
    """
    Normalise conf (max(conf_row, conf_col)), puis ajoute:
      - alpha * max(coin haut-droit) sur toutes les cases i<j
      - alpha * max(coin bas-gauche) sur toutes les cases i>j
    Renvoie (conf_normalisée, M_mise_à_jour, u_max, l_max).
    """
    conf_raw = get_confusion_matrix(logits, target, use_logits, ignore_index) \
                 .to(M.device).to(torch.float32)

    row_sum = conf_raw.sum(dim=1, keepdim=True).clamp_min(1e-12)
    col_sum = conf_raw.sum(dim=0, keepdim=True).clamp_min(1e-12)
    conf_row = conf_raw / row_sum
    conf_col = conf_raw / col_sum
    conf = torch.maximum(conf_row, conf_col)

    C = conf.size(0)
    eff = conf if weights is None else (conf * weights.to(conf.device, conf.dtype))

    upper_mask = torch.triu(torch.ones(C, C, device=conf.device, dtype=torch.bool), diagonal=1)
    lower_mask = torch.tril(torch.ones(C, C, device=conf.device, dtype=torch.bool), diagonal=-1)

    u_max = eff[upper_mask].max().item() if upper_mask.any() else 0.0
    l_max = eff[lower_mask].max().item() if lower_mask.any() else 0.0

    per_cell = torch.zeros_like(conf)
    per_cell[upper_mask] = u_max
    per_cell[lower_mask] = l_max
    per_cell.fill_diagonal_(0.0)

    M = M.to(per_cell.device, per_cell.dtype)
    M = M + alpha * per_cell
    M.fill_diagonal_(0.0)
    return conf, M


# ---------------------- Test minimal ----------------------
torch.manual_seed(0)
N, C = 8, 3

# logits artificiels (N, C) -> (N, C, ...) attendu par nos fonctions
raw = torch.tensor([
    [ 5.0,  1.0, -2.0],  # préd = 0
    [ 0.5,  2.0,  0.0],  # préd = 1
    [-1.0, -0.5,  3.0],  # préd = 2
    [ 4.0,  0.2,  0.1],  # préd = 0
    [ 0.1,  0.0,  0.2],  # préd = 2
    [ 0.2,  0.3,  0.1],  # préd = 1
    [ 0.2,  0.1,  0.3],  # préd = 2
    [ 2.0,  1.9,  1.8],  # préd = 0
])
logits = raw.unsqueeze(-1)  # (N, C, 1) pour montrer que ça marche avec dims en plus

# cibles (0..C-1)
target = torch.tensor([0, 1, 2, 0, 1, 2, 1, 0])

# M initiale à 0
M0 = torch.zeros((C, C), dtype=torch.float32)

# Confusion + update
conf = get_confusion_matrix(logits, target, use_logits=True)
print("Confusion brute:\n", conf)

conf2, M1 = confusion_matrix_update_corners(M0, logits, target, alpha=0.1, use_logits=True)
print("\nConfusion (renvoyée par update):\n", conf2)
print("\nM mise à jour:\n", M1)

Confusion brute:
 tensor([[3, 0, 0],
        [0, 1, 2],
        [0, 1, 1]])

Confusion (renvoyée par update):
 tensor([[1.0000, 0.0000, 0.0000],
        [0.0000, 0.5000, 0.6667],
        [0.0000, 0.5000, 0.5000]])

M mise à jour:
 tensor([[0.0000, 0.0667, 0.0667],
        [0.0500, 0.0000, 0.0667],
        [0.0500, 0.0500, 0.0000]])


In [39]:
import numpy as np
arrays = np.asarray([np.random.rand(1) for t in range(5)])
arrays

array([[0.11387302],
       [0.68931023],
       [0.66275561],
       [0.85212188],
       [0.62813478]])

In [43]:
import torch

arr = torch.nn.functional.softmax(torch.as_tensor(arrays), dim=0)

In [45]:
temperature = 1
arrays = arrays / temperature

arr = torch.nn.functional.softmax(torch.as_tensor(arrays), dim=0)
arr

tensor([[0.1209],
        [0.2149],
        [0.2093],
        [0.2529],
        [0.2021]], dtype=torch.float64)

In [46]:
temperature = 1.5
arrays = arrays / temperature

arr = torch.nn.functional.softmax(torch.as_tensor(arrays), dim=0)
arr

tensor([[0.1438],
        [0.2110],
        [0.2073],
        [0.2352],
        [0.2026]], dtype=torch.float64)

In [50]:
temperature = 10
arrays = arrays / temperature

arr = torch.nn.functional.softmax(torch.as_tensor(arrays), dim=0)
arr

tensor([[0.1997],
        [0.2001],
        [0.2000],
        [0.2002],
        [0.2000]], dtype=torch.float64)